**This notebook is an exercise in the [Intro to Game AI and Reinforcement Learning](https://www.kaggle.com/learn/intro-to-game-ai-and-reinforcement-learning) course.  You can reference the tutorial at [this link](https://www.kaggle.com/alexisbcook/n-step-lookahead).**

---


# Introduction

In the tutorial, you learned how to build a reasonably intelligent agent with the minimax algorithm.  In this exercise, you will check your understanding and submit your own agent to the competition.

In [ ]:
from learntools.core import binder
binder.bind(globals())
from learntools.game_ai.ex3 import *

### 1) A closer look

The heuristic from the tutorial looks at all groups of four adjacent grid locations on the same row, column, or diagonal and assigns points for each occurrence of the following patterns:

<center>
<img src="https://storage.googleapis.com/kaggle-media/learn/images/3NvBEGL.png" width=70%><br/>
</center>

Is it really necessary to use so many numbers to define the heuristic?  Consider simplifying it, as in the image below.

<center>
<img src="https://storage.googleapis.com/kaggle-media/learn/images/grViegG.png" width=70%><br/>
</center>

How would each heuristic score the potential moves in the example below (where, in this case, the agent looks only one step ahead)?  Which heuristic would lead to the agent selecting the better move?

<center>
<img src="https://storage.googleapis.com/kaggle-media/learn/images/LWPLy7N.png" width=100%><br/>
</center>

In [ ]:
#q_1.hint()

In [ ]:
# Check your answer (Run this code cell to receive credit!)
q_1.solution()

### 2) Count the leaves

In the tutorial, we worked with a small game tree.

<center>
<img src="https://storage.googleapis.com/kaggle-media/learn/images/BrRe7Bu.png" width=90%><br/>
</center>

The game tree above has 8 leaf nodes that appear at the bottom of the tree.  By definition, "leaf nodes" in a game tree are nodes that don't have nodes below them.

In the ConnectX competition, the game trees will be much larger!  

To see this, consider a minimax agent that is trying to plan its first move, where all columns in the game board are  empty.  Say the agent builds a game tree of depth 3.  How many leaf nodes are in the game tree?  

Use your answer to fill in the blank below.

In [ ]:
# Depth 3, every column legal at every ply on an empty board:
# 7 agent moves, 7 opponent replies to each, 7 agent replies to each of those.
num_leaves = 7*7*7

# Check your answer
q_2.check()


In [ ]:
# Lines below will give you a hint or solution code
#q_2.hint()
#q_2.solution()

### 3) Which move will the agent select?

In this question, you'll check your understanding of the minimax algorithm.  Remember that with this algorithm, 
> The agent chooses moves to get a score that is as high as possible, and it assumes the opponent will counteract this by choosing moves to force the score to be as low as possible.

Consider the toy example below of a game tree that the agent will use to select its next move.  
<center>
<img src="https://storage.googleapis.com/kaggle-media/learn/images/QlfWGM9.png" width=80%><br/>
</center>

Which move will the agent select?  Use your answer to set the value of the `selected_move` variable below.  Your answer should be one of `1`, `2`, or `3`.

In [ ]:
# Score each move by the WORST leaf under it, not the best -- the opponent
# picks the reply, and it picks the one that hurts most. Then take the move
# whose worst case is highest. Move 3 has the best worst case.
selected_move = 3

# Check your answer
q_3.check()


In [ ]:
# Lines below will give you a hint or solution code
#q_3.hint()
#q_3.solution()

### 4) Examine the assumptions

The minimax agent assumes that its opponent plays optimally (with respect to the heuristic, and using a game tree of limited depth).  But this is almost never the case, in practice: it's far more likely for the agent to encounter a suboptimal (that is: worse than optimal) opponent.  

Say the minimax agent encounters a suboptimal opponent. Should we expect the minimax agent to still play the game well, despite the contradiction with its assumptions?  If so, why?

In [ ]:
#q_4.hint()

In [ ]:
# Check your answer (Run this code cell to receive credit!)
q_4.solution()

### 5) Submit to the competition

Now, it's time to submit an agent to the competition!  Use the next code cell to define an agent.  (You can see an example of how to write a valid agent in **[this notebook](https://www.kaggle.com/alexisbcook/create-a-connectx-agent)**.)

If you decide to use the minimax code from the tutorial, you might like to add [**alpha-beta pruning**](https://en.wikipedia.org/wiki/Alpha%E2%80%93beta_pruning) to decrease the computation time (i.e., get the minimax algorithm to run much faster!).  In this case, "alpha" and "beta" to refer to two values that are maintained while the algorithm is running, that help to identify early stopping conditions.  

Without alpha-beta pruning, minimax evaluates each leaf node.  With alpha-beta pruning, minimax only evaluates nodes that could provide information that affects the agent's choice of action.  Put another way, it identifies nodes that could not possibly affect the final result and avoids evaluating them.

In [ ]:
def my_agent(obs, config):
    import random
    import time

    t_start = time.time()

    try:
        COLS = config.columns
        ROWS = config.rows
        K = config.inarow

        board = obs.board
        me = obs.mark
        valid = [c for c in range(COLS) if board[c] == 0]
        if not valid:
            return 0

        # --- bitboard layout -------------------------------------------
        # bit index = col * H + row, row 0 = bottom. H = ROWS + 1 leaves one
        # always-empty sentinel row per column, which is what stops vertical
        # and diagonal runs from wrapping between columns.
        H = ROWS + 1
        DIRS = (1, H, H + 1, H - 1)

        FULL = 0
        BOTTOM = 0
        COLMASK = []
        for c in range(COLS):
            cm = 0
            for r in range(ROWS):
                cm |= 1 << (c * H + r)
            COLMASK.append(cm)
            FULL |= cm
            BOTTOM |= 1 << (c * H)

        my_pos = 0
        mask = 0
        for c in range(COLS):
            for r in range(ROWS):
                cell = board[(ROWS - 1 - r) * COLS + c]
                if cell:
                    b = 1 << (c * H + r)
                    mask |= b
                    if cell == me:
                        my_pos |= b
        op_pos = mask ^ my_pos

        # Centre columns first: they sit in more winning lines, so they produce
        # cutoffs earlier.
        mid = (COLS - 1) / 2.0
        ORDER = sorted(range(COLS), key=lambda c: abs(c - mid))

        # Shift plans for "which empty squares complete a run of K".
        # For each direction and each position of the gap within the run, the
        # other K-1 cells must already be ours.
        PLANS = []
        for d in DIRS:
            for gap in range(K):
                PLANS.append(tuple((i - gap) * d for i in range(K) if i != gap))

        def shift(x, s):
            return x >> s if s >= 0 else x << -s

        def connected(pos):
            for d in DIRS:
                m = pos
                for i in range(1, K):
                    m &= pos >> (d * i)
                    if not m:
                        break
                if m:
                    return True
            return False

        def threat_squares(pos, msk):
            """Empty cells that would complete a K-run for `pos`."""
            res = 0
            for plan in PLANS:
                m = shift(pos, plan[0])
                for s in plan[1:]:
                    m &= shift(pos, s)
                    if not m:
                        break
                if m:
                    res |= m
            return res & FULL & ~msk

        CENTRE = COLMASK[COLS // 2]
        if COLS % 2 == 0:
            CENTRE |= COLMASK[COLS // 2 - 1]

        def evaluate(my, op, msk):
            my_t = threat_squares(my, msk)
            op_t = threat_squares(op, msk)
            playable = (msk + BOTTOM) & FULL
            score = 0
            score += 16 * bin(my_t).count("1") - 16 * bin(op_t).count("1")
            score += 40 * bin(my_t & playable).count("1")
            score -= 40 * bin(op_t & playable).count("1")
            score += 2 * bin(my & CENTRE).count("1")
            score -= 2 * bin(op & CENTRE).count("1")
            return score

        # --- search -----------------------------------------------------
        WIN = 10 ** 6
        INF = 10 ** 9
        budget = 0.90
        counter = [0]
        tt = {}

        class TimeUp(Exception):
            pass

        def negamax(my, op, msk, depth, alpha, beta, ply):
            counter[0] += 1
            if counter[0] & 511 == 0 and time.time() - t_start > budget:
                raise TimeUp

            nb = msk + BOTTOM
            moves = []
            for c in ORDER:
                b = nb & COLMASK[c]
                if b & FULL:
                    if connected(my | b):
                        return WIN - ply
                    moves.append(b)
            if not moves:
                return 0
            if depth == 0:
                return evaluate(my, op, msk)

            key = (my, msk)
            hit = tt.get(key)
            if hit is not None and hit[0] >= depth:
                _, flag, val = hit
                if flag == 0:
                    return val
                if flag == 1 and val > alpha:
                    alpha = val
                elif flag == 2 and val < beta:
                    beta = val
                if alpha >= beta:
                    return val

            alpha0 = alpha
            best = -INF
            for b in moves:
                v = -negamax(op, my | b, msk | b, depth - 1, -beta, -alpha, ply + 1)
                if v > best:
                    best = v
                if best > alpha:
                    alpha = best
                if alpha >= beta:
                    break

            if best <= alpha0:
                flag = 2
            elif best >= beta:
                flag = 1
            else:
                flag = 0
            tt[key] = (depth, flag, best)
            return best

        # --- root -------------------------------------------------------
        nb0 = mask + BOTTOM
        root_moves = []
        for c in ORDER:
            b = nb0 & COLMASK[c]
            if b & FULL:
                root_moves.append((c, b))

        # Win now, and never hand the opponent a win next move.
        for c, b in root_moves:
            if connected(my_pos | b):
                return c
        for c, b in root_moves:
            if connected(op_pos | b):
                return c

        best_move = root_moves[0][0]
        max_depth = ROWS * COLS - bin(mask).count("1")
        depth = 2
        while depth <= max_depth:
            try:
                alpha = -INF
                local_best = None
                for c, b in root_moves:
                    v = -negamax(op_pos, my_pos | b, mask | b, depth - 1, -INF, -alpha, 1)
                    if local_best is None or v > alpha:
                        alpha = v
                        local_best = c
                if local_best is not None:
                    best_move = local_best
                if alpha >= WIN - depth:
                    break
            except TimeUp:
                break
            if time.time() - t_start > budget * 0.5:
                break
            depth += 1

        return int(best_move)

    except Exception:
        try:
            return int(random.choice([c for c in range(config.columns) if obs.board[c] == 0]))
        except Exception:
            return 0


In [ ]:
# Run this code cell to get credit for creating an agent
q_5.check()

In [ ]:
import inspect
import os

def write_agent_to_file(function, file):
    with open(file, "a" if os.path.exists(file) else "w") as f:
        f.write(inspect.getsource(function))
        print(function, "written to", file)

write_agent_to_file(my_agent, "submission.py")

Then, follow these steps to submit your agent to the competition:
1. Begin by clicking on the **Save Version** button in the top right corner of the window.  This will generate a pop-up window.  
2. Ensure that the **Save and Run All** option is selected, and then click on the **Save** button.
3. This generates a window in the bottom left corner of the notebook.  After it has finished running, click on the number to the right of the **Save Version** button.  This pulls up a list of versions on the right of the screen.  Click on the ellipsis **(...)** to the right of the most recent version, and select **Open in Viewer**.  This brings you into view mode of the same page. You will need to scroll down to get back to these instructions.
4. Click on the **Data** tab near the top of the screen.  Then, click on the file you would like to submit, and click on the **Submit** button to submit your results to the leaderboard.

You have now successfully submitted to the competition!

If you want to keep working to improve your performance, select the **Edit** button in the top right of the screen. Then you can change your code and repeat the process. There's a lot of room to improve, and you will climb up the leaderboard as you work.


Go to **"My Submissions"** to view your score and episodes being played.

# Keep going

Move on to learn how to **[use deep reinforcement learning](https://www.kaggle.com/alexisbcook/deep-reinforcement-learning)** to develop an agent without a heuristic!

---




*Have questions or comments? Visit the [course discussion forum](https://www.kaggle.com/learn/intro-to-game-ai-and-reinforcement-learning/discussion) to chat with other learners.*